In [1]:
import json

def Load_josn(p):
    with open(p, 'r', encoding='utf-8') as f:
        d = json.load(f)
    return d


pdf_urls=Load_josn('data_set\\pdf_urls.json')
question_doc = Load_josn('data_set\\qrels.json')
queries=Load_josn('data_set\\queries.json')
answers=Load_josn('data_set\\answers.json')



In [2]:
import pandas as pd
df = pd.DataFrame.from_dict(question_doc, orient='index')
df.index.name = 'query_id'
df = df.reset_index()  
df.head(2)

,query_id,doc_id,section_id
0,852703f0-8373-43a2-a18a-eb5908ad0779,2410.14077v2,1
1,9199173b-3ed1-4118-88cd-1713fc5fa8a7,2404.00822v2,17


In [3]:
doc_list=df.groupby('doc_id').count().sort_values(by='query_id',ascending=False).head(10).index.tolist()
df=df[df['doc_id'].isin(doc_list)]

### To Download the PDFs use the below block

In [4]:
# import requests, os
# from concurrent.futures import ThreadPoolExecutor, as_completed

# os.makedirs("pdfs", exist_ok=True)
# items ={}
# for i ,j in pdf_urls.items():
#     if(i in doc_list):
#         items[i]=j
    
# def download(name_url):
#     name, url = name_url
#     try:
#         fname = os.path.join("pdfs", f"{name}.pdf")
        
#         res = requests.get(url, timeout=30)
#         res.raise_for_status()
#         with open(fname, "wb") as f:
#             f.write(res.content)
#         return name, True, None
#     except Exception as e:
#         return name, False, str(e)

# pdfs_ = []
# with ThreadPoolExecutor(max_workers=8) as executor:
#     futures = [executor.submit(download, item) for item in items.items()]
#     for future in as_completed(futures):
#         name, ok, err = future.result()
#         if ok:
#             pdfs_.append(name)
#         else:
#             print(f"failed: {name} -> {err}")

# print(f"{len(pdfs_)}/{len(items)} downloaded")

In [5]:
q_df = pd.DataFrame.from_dict(queries, orient='index')  # columns: query, type, source
q_df.index.name = 'query_id'
q_df = q_df.reset_index().rename(columns={'type': 'query_type', 'source': 'query_source', 'query': 'query'})

df = df.merge(q_df, on='query_id', how='left')

In [6]:
answer_df=pd.DataFrame.from_dict(answers,orient="index")
answer_df.index.name='query_id'
df=df.merge(answer_df,on='query_id',how='left')

In [7]:
df=df.rename(columns={0:'answer'})
df.head(2)


,query_id,doc_id,section_id,query,query_type,query_source,answer
0,dc064d11-cd18-4866-8a99-f16b0abec9c6,2401.07294v4,12,How does the MLMM approach affect the analysis...,abstractive,text-image,The MLMM approach affects the analysis of RMSE...
1,bac61451-d99a-43b3-9754-b8a593e5d1d7,2401.11899v3,10,Does bounded invariance affect how probability...,extractive,text,"No, bounded invariance states that changes in ..."


---
## Configuration

In [8]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from pydantic import BaseModel
from typing import Literal

# ── Paths ──
PDF_DIR = "pdfs"
TEST_DOC = "2401.03305v2"
PDF_PATH = os.path.join(PDF_DIR, f"{TEST_DOC}.pdf")
TOP_K = 5

# ── Torch compile fix for Windows (Triton not available) ──
os.environ["TORCHDYNAMO_DISABLE"] = "1"

---
## Pydantic model for LLM judge

In [9]:
class RetrievalJudgement(BaseModel):
    """Structured output schema for the retrieval judge."""
    result: Literal["PASS", "FAIL"]
    reason: str

---
## Judge prompts

In [10]:
RETRIEVAL_JUDGE_SYSTEM_PROMPT = (
    "You are an evaluator for a Retrieval-Augmented Generation (RAG) system. "
    "Your task is to determine whether the retrieved chunks contain sufficient "
    "and relevant information to answer the given query correctly.\n\n"
    "You will receive:\n"
    "* A QUERY: the question asked by the user.\n"
    "* A GROUND TRUTH ANSWER: the expected answer.\n"
    "* RETRIEVED CHUNKS: the documents/chunks retrieved by the RAG system.\n\n"
    "Evaluate the retrieved chunks as a whole.\n\n"
    "Return:\n"
    "PASS if the retrieved chunks contain enough relevant information to "
    "derive the ground-truth answer.\n"
    "FAIL if:\n"
    "* The retrieved chunks are irrelevant to the query.\n"
    "* The chunks do not contain the information required to answer the query.\n"
    "* Important information required for the ground-truth answer is missing.\n"
    "* The chunks contain information that contradicts the ground-truth answer "
    "and would prevent a correct answer.\n"
    "* The information is too vague or incomplete to reasonably derive the expected answer.\n\n"
    "Important rules:\n"
    "* Judge semantic relevance, not just keyword overlap.\n"
    "* Do not use external knowledge.\n"
    "* Do not assume information that is not present in the retrieved chunks.\n"
    "* The ground-truth answer is the reference for determining whether the "
    "retrieved information is sufficient.\n"
    "* The retrieved chunks do not need to contain the exact wording of the "
    "ground-truth answer.\n"
    "* Paraphrases and semantically equivalent information should be considered valid.\n"
    "* Multiple chunks can collectively provide the required information.\n"
    "* Do not judge the quality of the chunking method itself.\n"
    "* Judge only whether the retrieved content is sufficient for answering the query."
)

RETRIEVAL_JUDGE_USER_TEMPLATE = (
    "Evaluate the following RAG retrieval result.\n\n"
    "QUERY:\n{query}\n\n"
    "GROUND TRUTH ANSWER:\n{ground_truth_answer}\n\n"
    "RETRIEVED CHUNKS:\n{retrieved_chunks}\n\n"
    "Determine whether the retrieved chunks are sufficient to answer the "
    "query and derive the ground-truth answer."
)

---
## Helper functions

In [11]:
from rag_eval.embedders import HuggingFaceEmbedder

embedder = HuggingFaceEmbedder()


def parse_and_chunk(parser, chunker, pdf_path):
    """Parse a PDF and chunk it. Returns list of chunk dicts."""
    doc = parser.parse(pdf_path)
    chunks = chunker.chunk(doc)
    chunk_data = []
    for c in chunks:
        chunk_data.append({
            "chunk_no": c.chunk_no,
            "text": c.text,
            "page_numbers": c.page_numbers,
            "embedding": embedder.embed_query(c.text),
        })
    return chunk_data


def retrieve_top_k(query_embedding, chunks_df, k=TOP_K):
    """Cosine similarity retrieval. Returns top-k chunk dicts."""
    chunk_embs = np.vstack(chunks_df["embedding"].values)
    query_emb = np.array(query_embedding).reshape(1, -1)
    sims = cosine_similarity(query_emb, chunk_embs)[0]
    top_idx = np.argsort(sims)[-k:][::-1]
    results = []
    for idx in top_idx:
        row = chunks_df.iloc[idx]
        results.append({
            "text": row["text"],
            "page_numbers": row["page_numbers"],
            "similarity": float(sims[idx]),
            "chunk_no": row["chunk_no"],
        })
    return results


def format_chunks_for_judge(retrieved):
    """Format retrieved chunks into a string for the judge prompt."""
    parts = []
    for i, c in enumerate(retrieved, 1):
        pages = c["page_numbers"] if c["page_numbers"] else "?"
        parts.append(f"[Chunk {i}, pages {pages}]\n{c['text']}")
    return "\n\n---\n\n".join(parts)


def run_evaluation_pipeline(parser_name, chunks_df, query_df, judge_llm):
    """Run full retrieval evaluation for one parser pipeline.

    For each query:
      1. Retrieve top-k chunks via cosine similarity
      2. Ask the LLM judge: PASS or FAIL?
      3. Track retrieved pages for hit rate analysis
    """
    results = []

    for idx, row in query_df.iterrows():
        query_text = row["query"]
        gt_answer = row["answer"]
        query_emb = row["query_embedding"]

        # Retrieve
        retrieved = retrieve_top_k(query_emb, chunks_df, k=TOP_K)

        # Collect retrieved pages
        retrieved_pages = set()
        for c in retrieved:
            pn = c["page_numbers"]
            if pn:
                if isinstance(pn, list):
                    retrieved_pages.update(pn)
                else:
                    retrieved_pages.add(pn)

        # Format for judge
        chunks_str = format_chunks_for_judge(retrieved)

        # LLM Judge call
        user_prompt = RETRIEVAL_JUDGE_USER_TEMPLATE.format(
            query=query_text,
            ground_truth_answer=gt_answer,
            retrieved_chunks=chunks_str,
        )

        try:
            verdict = judge_llm.invoke_structured(
                prompt=user_prompt,
                response_model=RetrievalJudgement,
                system_prompt=RETRIEVAL_JUDGE_SYSTEM_PROMPT,
            )
            result_val = verdict.result
            reason_val = verdict.reason
        except Exception as e:
            result_val = "ERROR"
            reason_val = str(e)

        results.append({
            "query_id": row["query_id"],
            "query": query_text,
            "gt_answer": gt_answer,
            "result": result_val,
            "reason": reason_val,
            "top_similarity": retrieved[0]["similarity"] if retrieved else 0,
            "retrieved_pages": sorted(retrieved_pages),
            "num_chunks": len(retrieved),
        })

        # Progress indicator
        status = result_val
        print(f"  [{status}] {query_text[:80]}")

    return pd.DataFrame(results)


def compute_metrics(results_df):
    """Compute precision, recall, hit rate from judge verdicts.

    Each query is one retrieval event judged by the LLM:
      PASS = retrieved context is sufficient (true positive)
      FAIL = retrieved context is insufficient (false negative)

    precision@k : PASS / (PASS + FAIL) = fraction of retrievals sufficient
    recall@k    : PASS / total answerable queries (all have GT, so = total)
    hit_rate    : same as precision here (judge evaluates full top-k set)
    """
    valid = results_df[results_df["result"].isin(["PASS", "FAIL"])]
    total = len(valid)

    if total == 0:
        return {"precision@k": 0, "recall@k": 0, "hit_rate": 0,
                "pass": 0, "fail": 0, "total": 0, "errors": len(results_df)}

    passes = int((valid["result"] == "PASS").sum())
    fails = int((valid["result"] == "FAIL").sum())
    errors = int(len(results_df) - total)

    precision = passes / total
    recall = passes / total
    hit_rate = passes / total

    return {
        "precision@k": round(precision, 4),
        "recall@k": round(recall, 4),
        "hit_rate": round(hit_rate, 4),
        "pass": passes,
        "fail": fails,
        "total": total,
        "errors": errors,
    }

d:\projects\RAG-Chunking-Eval\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3952.00it/s]


---
## Filter queries for the test document

In [12]:
test_queries = df[df["doc_id"] == TEST_DOC].copy()
print(f"Queries for {TEST_DOC}: {len(test_queries)}")
test_queries.head(3)

Queries for 2401.03305v2: 10


,query_id,doc_id,section_id,query,query_type,query_source,answer
6,0920cb6c-229b-4b46-b2ab-834dffea6689,2401.03305v2,2,How do implementation shortfall (IS) and targe...,abstractive,text,Implementation shortfall (IS) orders aim to ex...
8,fd97f39d-243e-4075-b04b-2f2f459db403,2401.03305v2,9,What are the key differences between implement...,abstractive,text-image,Implementation shortfall (IS) involves executi...
19,782b73e6-acc4-43ed-a6ae-97a9f4039769,2401.03305v2,2,What challenges do large position holders face...,abstractive,text,"Large position holders, such as pension funds ..."


---
## Embed all queries (done once, shared by both pipelines)

In [13]:
test_queries["query_embedding"] = test_queries["query"].apply(embedder.embed_query)
print(f"Embedded {len(test_queries)} queries")

Embedded 10 queries


---
## Initialize Ollama LLM judge

In [18]:
from rag_eval.llms import OllamaLLM

judge_llm = OllamaLLM(model="gemma3:4b", temperature=0.0)

# Quick sanity check
test_result = judge_llm.invoke("Say hello in one word.")
print(f"Ollama is alive: {test_result}")

Ollama is alive: Hello!



---
---
# Pipeline 1: PDFPlumber + RecursiveChunker

### 1a. Parse & chunk

In [19]:
from rag_eval.parsers import PDFPlumberParser
from rag_eval.chunkers import RecursiveChunker

pdfplumber_parser = PDFPlumberParser()
recursive_chunker = RecursiveChunker()

pdfplumber_chunks = parse_and_chunk(pdfplumber_parser, recursive_chunker, PDF_PATH)
pdfplumber_chunks_df = pd.DataFrame(pdfplumber_chunks)

print(f"PDFPlumber chunks: {len(pdfplumber_chunks_df)}")
print(f"Sample: {pdfplumber_chunks_df.iloc[0]['text'][:150]}...")

PDFPlumber chunks: 187
Sample: Leveraging IS and TC: Optimal order execution subject to
reference strategies
Xue Cheng1, Peng Guo1, and Tai-Ho Wang2
1
LMEQF,DepartmentofFinancialMat...


### 1b. Run retrieval evaluation

In [20]:
print(f"Evaluating {len(test_queries)} queries against PDFPlumber chunks...\n")

pdfplumber_results = run_evaluation_pipeline(
    parser_name="pdfplumber",
    chunks_df=pdfplumber_chunks_df,
    query_df=test_queries,
    judge_llm=judge_llm,
)

pdfplumber_results[["query", "result", "reason"]].head(5)

Evaluating 10 queries against PDFPlumber chunks...

  [PASS] How do implementation shortfall (IS) and target close (TC) orders differ in trad
  [PASS] What are the key differences between implementation shortfall and target close t
  [PASS] What challenges do large position holders face when executing trades in financia
  [PASS] What role does parameter $\kappa$ play in shaping trading trajectories for IS an
  [PASS] What is a reference strategy in order execution brokerage?
  [FAIL] Why is stress testing important for financial strategies?
  [PASS] What role do trading trajectories play in optimal order execution strategies?
  [PASS] Is there overshooting in the optimal strategy if \( R < A - (x_0 - A)/(\cosh(\ka
  [FAIL] How does the optimal investment strategy compare to the TWAP strategy in terms o
  [FAIL] What impact do transaction costs have on aligning optimal trading strategies wit


,query,result,reason
0,How do implementation shortfall (IS) and targe...,PASS,The retrieved chunks provide sufficient inform...
1,What are the key differences between implement...,PASS,The retrieved chunks contain the necessary inf...
2,What challenges do large position holders face...,PASS,The retrieved chunks collectively provide suff...
3,What role does parameter $\kappa$ play in shap...,PASS,All the retrieved chunks contain relevant info...
4,What is a reference strategy in order executio...,PASS,All the retrieved chunks contain information r...


### 1c. Metrics

In [21]:
pdfplumber_metrics = compute_metrics(pdfplumber_results)
print("PDFPlumber + RecursiveChunker:")
for k, v in pdfplumber_metrics.items():
    print(f"  {k}: {v}")

PDFPlumber + RecursiveChunker:
  precision@k: 0.7
  recall@k: 0.7
  hit_rate: 0.7
  pass: 7
  fail: 3
  total: 10
  errors: 0


---
---
# Pipeline 2: Docling + RecursiveChunker

### 2a. Parse & chunk

In [22]:
from rag_eval.parsers import DoclingParser

docling_parser = DoclingParser(use_gpu=True)

# Reuse the same RecursiveChunker for a fair comparison —
# the only variable is the parser output quality
docling_chunks = parse_and_chunk(docling_parser, recursive_chunker, PDF_PATH)
docling_chunks_df = pd.DataFrame(docling_chunks)

print(f"Docling chunks: {len(docling_chunks_df)}")
print(f"Sample: {docling_chunks_df.iloc[0]['text'][:150]}...")

[INFO] 2026-08-16 23:59:12,202 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-16 23:59:12,216 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-08-16 23:59:12,240 [RapidOCR] download_file.py:60: File exists and is valid: D:\projects\RAG-Chunking-Eval\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-16 23:59:12,241 [RapidOCR] main.py:50: Using D:\projects\RAG-Chunking-Eval\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-16 23:59:12,436 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-16 23:59:12,437 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-08-16 23:59:12,441 [RapidOCR] download_file.py:60: File exists and is valid: D:\projects\RAG-Chunking-Eval\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-08-16 23:59:12,442 [RapidOCR] main.py:50: Using D:\projects\RAG-Chunking-Eval\.venv\Lib\site-packages\rapidocr\models\

Docling chunks: 178
Sample: 1
Leveraging IS and TC: Optimal order execution subject to reference strategies
Xue Cheng 1 , Peng Guo 1 , and Tai-Ho Wang 2
LMEQF, Department of Fina...


### 2b. Run retrieval evaluation

In [23]:
print(f"Evaluating {len(test_queries)} queries against Docling chunks...\n")

docling_results = run_evaluation_pipeline(
    parser_name="docling",
    chunks_df=docling_chunks_df,
    query_df=test_queries,
    judge_llm=judge_llm,
)

docling_results[["query", "result", "reason"]].head(5)

Evaluating 10 queries against Docling chunks...

  [PASS] How do implementation shortfall (IS) and target close (TC) orders differ in trad
  [FAIL] What are the key differences between implementation shortfall and target close t
  [PASS] What challenges do large position holders face when executing trades in financia
  [PASS] What role does parameter $\kappa$ play in shaping trading trajectories for IS an
  [PASS] What is a reference strategy in order execution brokerage?
  [PASS] Why is stress testing important for financial strategies?
  [PASS] What role do trading trajectories play in optimal order execution strategies?
  [PASS] Is there overshooting in the optimal strategy if \( R < A - (x_0 - A)/(\cosh(\ka
  [PASS] How does the optimal investment strategy compare to the TWAP strategy in terms o
  [PASS] What impact do transaction costs have on aligning optimal trading strategies wit


,query,result,reason
0,How do implementation shortfall (IS) and targe...,PASS,The retrieved chunks collectively provide the ...
1,What are the key differences between implement...,FAIL,The retrieved chunks discuss implementation sh...
2,What challenges do large position holders face...,PASS,The retrieved chunks collectively provide suff...
3,What role does parameter $\kappa$ play in shap...,PASS,All the chunks provide relevant information to...
4,What is a reference strategy in order executio...,PASS,The retrieved chunks collectively provide suff...


### 2c. Metrics

In [24]:
docling_metrics = compute_metrics(docling_results)
print("Docling + RecursiveChunker:")
for k, v in docling_metrics.items():
    print(f"  {k}: {v}")

Docling + RecursiveChunker:
  precision@k: 0.9
  recall@k: 0.9
  hit_rate: 0.9
  pass: 9
  fail: 1
  total: 10
  errors: 0


---
---
# Side-by-side comparison

In [25]:
comparison = pd.DataFrame({
    "PDFPlumber + Recursive": pdfplumber_metrics,
    "Docling + Recursive": docling_metrics,
}).T

comparison.index.name = "Pipeline"
comparison

,precision@k,recall@k,hit_rate,pass,fail,total,errors
Pipeline,,,,,,,
PDFPlumber + Recursive,0.7,0.7,0.7,7.0,3.0,10.0,0.0
Docling + Recursive,0.9,0.9,0.9,9.0,1.0,10.0,0.0


### Per-query diff: where do they disagree?

In [26]:
merged = pdfplumber_results[["query_id", "query", "result", "reason"]].merge(
    docling_results[["query_id", "result", "reason"]],
    on="query_id",
    suffixes=("_pdfplumber", "_docling"),
)

disagree = merged[merged["result_pdfplumber"] != merged["result_docling"]]
print(f"Disagreements: {len(disagree)} / {len(merged)} queries\n")

# Show the interesting cases where parsers diverge
disagree[["query", "result_pdfplumber", "reason_pdfplumber",
          "result_docling", "reason_docling"]]

Disagreements: 4 / 10 queries



,query,result_pdfplumber,reason_pdfplumber,result_docling,reason_docling
1,What are the key differences between implement...,PASS,The retrieved chunks contain the necessary inf...,FAIL,The retrieved chunks discuss implementation sh...
5,Why is stress testing important for financial ...,FAIL,The retrieved chunks discuss stress testing an...,PASS,The retrieved chunks collectively provide suff...
8,How does the optimal investment strategy compa...,FAIL,The retrieved chunks discuss the comparison of...,PASS,The retrieved chunks collectively provide suff...
9,What impact do transaction costs have on align...,FAIL,The retrieved chunks discuss transaction costs...,PASS,The retrieved chunks collectively provide the ...


### Visual comparison

In [ ]:
import matplotlib.pyplot as plt

metrics_to_plot = ["precision@k", "recall@k", "hit_rate"]
pdfp_vals = [pdfplumber_metrics[m] for m in metrics_to_plot]
docl_vals = [docling_metrics[m] for m in metrics_to_plot]

x = np.arange(len(metrics_to_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar(x - width/2, pdfp_vals, width, label="PDFPlumber", color="#5DCAA5")
bars2 = ax.bar(x + width/2, docl_vals, width, label="Docling", color="#7F77DD")

ax.set_ylabel("Score")
ax.set_title(f"Retrieval quality: PDFPlumber vs Docling (top-{TOP_K}, {TEST_DOC})")
ax.set_xticks(x)
ax.set_xticklabels([m.replace("@k", f"@{TOP_K}") for m in metrics_to_plot])
ax.set_ylim(0, 1.05)
ax.legend()

for bar in bars1 + bars2:
    h = bar.get_height()
    ax.annotate(f"{h:.2f}", xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 4), textcoords="offset points",
                ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()